<a href="https://colab.research.google.com/github/zwimpee/cursivetransformer/blob/main/cursivetransformer_mech_interp_part_3_stroke_attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Visualizing Attention on Stroke Tokens

# Setup

In [1]:
!pip install transformer_lens
!pip install gradio
!pip install wandb
!pip install einops
!pip install matplotlib
!pip install datasets

# Clone the cursivetransformer repository and install its requirements
!rm -rf cursivetransformer && git clone https://github.com/zwimpee/cursivetransformer.git
!pip install -r cursivetransformer/requirements.txt

INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.6/175.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.7/739.7 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.6/471.6 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 76.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.8/313.8 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7

In [2]:
import sys
sys.path.append('/content/cursivetransformer')
from cursivetransformer.model import get_all_args, get_checkpoint, get_latest_checkpoint_artifact
from cursivetransformer.data import create_datasets, offsets_to_strokes, strokes_to_offsets
from cursivetransformer.sample import generate, generate_n_words, plot_strokes
from cursivetransformer.mech_interp import (
    HookedCursiveTransformer,
    HookedCursiveTransformerConfig,
    convert_cursivetransformer_model_config,
    visualize_attention,
    generate_repeated_stroke_tokens,
    generate_random_ascii_context,
    run_and_cache_model_repeated_tokens,
    compute_induction_scores,
    plot_induction_scores,
    plot_head_attention_pattern,
    create_induction_summary,
    ablate_heads,
    get_induction_positions,
    compute_loss_on_induction_positions,
    visualize_attention_patterns
)

import pandas as pd
import os

import copy
import types
from typing import List, Callable, Dict, Optional, Union, Tuple
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import einops
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.io as pio
import circuitsvis as cv
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from jaxtyping import Float, Int


import transformer_lens.utils as utils
from transformer_lens.hook_points import HookPoint
from transformer_lens import ActivationCache

torch.set_grad_enabled(False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
# wandb_api_key - e56bbe426145e5983e72a933299daca195ebb6a7
args = get_all_args(False)
args.sample_only = True
args.max_seq_length = 1250
args.load_from_run_id = '7e9hz1og'
args.wandb_project = 'bigbank_2k'
args.wandb_entity = 'sam-greydanus'
args.dataset_name = 'bigbank'
args.wandb_run_name = 'cursivetransformer_mech_interp_part_3_stroke_attention'
torch.manual_seed(args.seed)
torch.cuda.manual_seed_all(args.seed)
train_dataset, test_dataset = create_datasets(args)
args.vocab_size = train_dataset.get_vocab_size()
args.block_size = train_dataset.get_stroke_seq_length()
args.context_block_size = train_dataset.get_text_seq_length()
args.context_vocab_size = train_dataset.get_char_vocab_size()

Enter your W&B API key: ··········
Trying to load dataset file from /content/cursivetransformer/data/bigbank.json.zip
Succeeded in loading the bigbank dataset; contains 2000 items.
For a dataset of 1900 examples we can generate 205257574037880 combinations of 5 examples.
Generating 497000 random combinations.
For a dataset of 100 examples we can generate 75287520 combinations of 5 examples.
Generating 3000 random combinations.
Number of examples in the train dataset: 497000
Number of examples in the test dataset: 3000
Max token sequence length: 1250
Number of unique characters in the ascii vocabulary: 71
Ascii vocabulary:
	" enaitoshrdx.vpukbgfcymzw1lqj804I92637OTAS5N)EHR"'(BCQLMWYU,ZF!DXV?KPGJ"
Split up the dataset into 497000 training examples and 3000 test examples


## Verify that the original and hooked versions of the model produce the same output logits for a given sequence

In [ ]:
# Load the original model
original_model, _, _, _, _ = get_checkpoint(args, sample_only=True)
original_model.eval()

In [ ]:
# Load the hooked model
cfg = convert_cursivetransformer_model_config(args)
hooked_model = HookedCursiveTransformer.from_pretrained("cursivetransformer", cfg)
hooked_model = hooked_model.to(device)
hooked_model.eval()

In [ ]:
# Prepare a sample input
sample_tokens = torch.randint(0, args.vocab_size, (1, args.block_size)).to(device)
sample_context = torch.randint(0, args.context_vocab_size, (1, args.context_block_size)).to(device)

In [ ]:
# Run both models
with torch.no_grad():
    original_output = original_model(sample_tokens, sample_context)
    hooked_output = hooked_model(sample_tokens, sample_context)

In [13]:
original_output = original_output[0]

In [14]:
# Compare outputs
max_diff = torch.max(torch.abs(original_output - hooked_output))
print(f"Maximum difference in logits: {max_diff.item()}")

if max_diff > 1e-5:
    print("Models are not equivalent!")
    # If models are not equivalent, let's print more detailed information
    print("Original model output shape:", original_output.shape)
    print("Hooked model output shape:", hooked_output.shape)
    print("Original model output mean:", original_output.mean().item())
    print("Hooked model output mean:", hooked_output.mean().item())
    print("Original model output std:", original_output.std().item())
    print("Hooked model output std:", hooked_output.std().item())
else:
    print("Models are equivalent.")

Maximum difference in logits: 109.5845718383789
Models are not equivalent!
Original model output shape: torch.Size([1, 1250, 382])
Hooked model output shape: torch.Size([1, 1250, 382])
Original model output mean: -14.839642524719238
Hooked model output mean: 0.018848156556487083
Original model output std: 14.006147384643555
Hooked model output std: 0.5719537138938904


Number of Transformer parameters: 379392
Model #params: 403840
Loaded model from local path: best_checkpoint.pt
Loading pretrained model cursivetransformer
Finding latest checkpoint for W&B run id 7e9hz1og
  model:best_checkpoint:v129
  model:best_checkpoint:v130
  model:best_checkpoint:v131
  model:best_checkpoint:v132
  model:best_checkpoint:v133
  model:best_checkpoint:v134
  model:best_checkpoint:v135
  model:best_checkpoint:v136
  model:best_checkpoint:v137
  model:best_checkpoint:v138
  model:best_checkpoint:v139
  model:best_checkpoint:v140
  model:best_checkpoint:v141
  model:best_checkpoint:v142
  model:best_checkpoint:v143
  model:best_checkpoint:v144
  model:best_checkpoint:v145
  model:best_checkpoint:v146
  model:best_checkpoint:v147
  model:best_checkpoint:v148
  wandb-history:run-7e9hz1og-history:v4
Selected:  model:best_checkpoint:v148


wandb:   1 of 1 files downloaded.  


Successfully loaded pretrained model cursivetransformer
Moving model to device:  cuda


TypeError: unsupported operand type(s) for -: 'tuple' and 'Tensor'

## Load model into HookedCursiveTransformer

In [ ]:
cfg = convert_cursivetransformer_model_config(args)
model = HookedCursiveTransformer.from_pretrained("cursivetransformer", cfg)

# Generate strokes using the hooked model

In [ ]:
text = "Hello, Sam!"
offset_samp, point_samp, attention_patterns = generate_n_words(model, test_dataset, text, n_words=2, is_hooked=True)

In [ ]:
if attention_patterns is not None:
    visualize_attention_patterns(attention_patterns)

In [ ]:
_ = plot_strokes(point_samp, text)